In [3]:
#load the best models from the checkpoints folder

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from keras.utils import to_categorical 
from tensorflow.keras.models import load_model, Model
import json
import helper
import plotly.express as px

MODEL_ID = 0

In [4]:
models = [load_model("./checkpoints/best_model_" + str(MODEL_ID) + ".h5")]
params = [{"group_size":1,"TEST_PROPORTION":0.25,"epochs":1000,"batch_size":256,"model":{"layers":[{"type":"input","shape":[26]},{"type":"dense","units":32,"activation":"relu"},{"type":"dense","units":16,"activation":"elu"},{"type":"dense","units":3,"activation":"linear","name":"Bottleneck"},{"type":"dense","units":16,"activation":"elu"},{"type":"dense","units":32,"activation":"relu"},{"type":"dense","units":26,"activation":"linear"}],"output_range":[-1,1]},"id":0}]

#load the json from params.json
# with open('params.json') as json_file:
#     params = json.load(json_file)
# models = [None]*len(params)
# for model_name in os.listdir("checkpoints"):
#     model = load_model("checkpoints/" + model_name)
#     model_id = int(model_name.split(".")[0].split("_")[-1])
#     models[model_id] = model

# print(models)

OSError: No file or directory found at ./checkpoints/best_model_0.h5

In [ ]:
#load the data

def avg(group, group_size): 
    group['GroupNumber'] = np.array(range(len(group.index))) // group_size
    res = group.groupby('GroupNumber').mean()
    return res
def load_data(group_size,output_range):
    data = pd.read_hdf("../../initialSingleCellDf-channel-20220916-MW_018-001.h5", key="df")
    #print all the columns
    ANTIGENS = ['null', 'E1', 'G4', 'V4', 'T4', 'Q4', 'A2', 'N4']
    data = data.loc[(data.index.get_level_values('CellType') == 'OT-1') & 
                       (data.index.get_level_values('Peptide').isin(ANTIGENS))
                    ]
    data = data.groupby(['Peptide', 'Time', 'Replicate', 'Concentration']).apply(avg, group_size=group_size)
    antigen = list(data.index.get_level_values('Peptide'))
    times = list(data.index.get_level_values('Time'))
    concentration = list(data.index.get_level_values('Concentration'))
    scaling_factor = 1024/(output_range[1]-output_range[0])
    X = np.array(data.values)/scaling_factor + output_range[0]
    y = np.array(list(map(lambda x: ANTIGENS.index(x), antigen)))
    y = to_categorical(y)
    return X,y,times,concentration

def get_model_embedding(model,params):
    X, y,times,concentration = load_data(params['group_size'],params['model']['output_range'])
    extractor = Model(inputs=model.inputs,
                        outputs=model.get_layer("Bottleneck").output)
    embedding = np.array(extractor(X))
    output = np.argmax(y,axis=1)
    ANTIGENS = ["null", "E1", "G4", "V4", "T4", "Q4", "A2", "N4"]
    output = np.array(ANTIGENS)[output]
    num_embedding_dimensions = embedding.shape[1]
    df = pd.DataFrame(embedding, columns=['Dimension '+str(i+1) for i in range(num_embedding_dimensions)])
    df["Antigen"] = output
    df["Time"] = times
    df["Concentration"] = concentration
    return df

In [ ]:
embedding = get_model_embedding(models[MODEL_ID],params[MODEL_ID])


KeyboardInterrupt: 

In [ ]:

def get_3d_plot(df,title,color='Antigen'):
    fig = px.scatter_3d(
        df,
        x='Dimension 1',
        y='Dimension 2',
        z='Dimension 3',
        color=color,
        title=title,
        # range_x=[-1,1],
        # range_y=[-1,1],
        # range_z=[-1,1]
    )

    fig.update_traces(marker=dict(size=4))
    fig.update_layout(legend_title="Antigen")
    fig.show()

def get_2d_plot(df,title,show_time=False):
    color = 'Antigen'
    if show_time:
        color = 'Time'
    fig = px.scatter(
        df,
        x='Dimension 1',
        y='Dimension 2',
        color=color,
        title=title,
        range_x=[-1,1],
        range_y=[-1,1],
    )

    fig.update_traces(marker=dict(size=4))
    fig.update_layout(legend_title="Antigen")
    fig.show()

In [ ]:

ANTIGENS = ["null", "E1", "G4", "V4", "T4", "Q4", "A2", "N4"]

# for i in range(len(ANTIGENS)):
#    #plot the data for each antigen
#     antigen = ANTIGENS[i]
#     antigen_df = embedding[embedding["Antigen"] == antigen]
#     get_3d_plot(antigen_df,"Autoencoder 3D Embedding Averaged Data (Validation Set) for " + antigen,color='Time')

get_3d_plot(embedding,"Autoencoder 3D Embedding Averaged Data (Validation Set) for All Antigens",color="Antigen")


In [ ]:
grouped = embedding.groupby(["Time","Antigen","Concentration"]).mean().reset_index()

In [ ]:
#group the data by time, concentration, and antigen
from scipy.optimize import curve_fit


def get_speeds_and_displacements(grouping, antigen,concentration):
    data = grouping[grouping["Antigen"] == antigen]
    data = data[data["Concentration"] == concentration]
    times = list(data["Time"].unique())
    data = data.to_dict('list')
    speeds = []
    distances = []
    starting_pos = np.array([data['Dimension 1'][0],data['Dimension 2'][0],data['Dimension 3'][0]])
    for i in range(len(times)-1):
        #plot the data for each antigen
        pos = np.array([data['Dimension 1'][i],data['Dimension 2'][i],data['Dimension 3'][i]])
        pos2 = np.array([data['Dimension 1'][i+1],data['Dimension 2'][i+1],data['Dimension 3'][i+1]])
        vel = pos2-pos
        speed = np.linalg.norm(vel)#/(times[i+1]-times[i])
        distances.append(np.linalg.norm(pos2-starting_pos))
        speeds.append(speed)
    return times,speeds,distances

def sigmoid(x, L ,x0, k, b):
        y = L / (1 + np.exp(-k*(x-x0)))+b
        return (y)
def fit_sigmoid(x,y):
    #fit a sigmoid to the data
    p0 = [max(y), np.median(x),1,min(y)] # this is an mandatory initial guess
    popt, pcov = curve_fit(sigmoid, x, y,p0,maxfev=10000,bounds=([0,-np.inf,0,-np.inf],[2*np.max(y),np.max(x),np.inf,np.inf]))
    return popt

# grouped=grouped[grouped["Concentration"]=='10pM']
# grouped=grouped[grouped["Antigen"]=='N4']
concentrations = list(grouped["Concentration"].unique())
antigens = list(grouped["Antigen"].unique())
print(antigens,concentrations)
antigen_strengths = {}
antigen_speeds = {}
for antigen in antigens:
    theoretical_max_vals = []
    theoretical_speeds = []
    actual_max_vals = []
    for concentration in concentrations:
        # print(antigen,concentration)
        times,speeds,distances = get_speeds_and_displacements(grouped,antigen,concentration)
        # sigmoid_fit =fit_sigmoid(times[:-1],distances)
        # print(f'antigen: {antigen} {concentration} speed:{sigmoid_fit[2]:.3f} theoretical max val:{sigmoid_fit[0]+sigmoid_fit[3]:.3f} actual max val:{max(distances):.3f} {sigmoid_fit[1]:.3f}')
        # print(sigmoid_fit)
        # theoretical_speeds.append(sigmoid_fit[2])
        # theoretical_max_vals.append(sigmoid_fit[0]+sigmoid_fit[3])
        actual_max_vals.append(max(distances))
        # plt.plot(times[:-1],distances)
        # plt.plot(times,sigmoid(times, *sigmoid_fit), label='fit')
        #plot the laplace smoothed data
    # max_avg = np.avg(theoretical_max_vals)
    # speed_avg = np.avg(theoretical_speeds)
    # antigen_speeds[antigen] = speed_avg
    antigen_strengths[antigen] = np.max(actual_max_vals)
    # plt.legend(concentrations)
    # plt.title("Distances for " + antigen)
    # #y range from 0-1
    # plt.ylim(0,7)
    # plt.show()


In [ ]:
#print the antigen strengths in order of strongest to weakest

ANTIGENS = ['N4', 'A2', 'Q4','T4',  'V4', 'G4', 'E1', 'null']
strengths = [antigen_strengths[antigen] for antigen in ANTIGENS]
speeds = [antigen_speeds[antigen] for antigen in ANTIGENS]
#normaize vs the null
strengths = np.array(strengths)/antigen_strengths['null']
#Bar plot from strongest to weakest
plt.bar(ANTIGENS,strengths)
plt.title("Antigen Strengths")
plt.show()

#plot the speeds
plt.bar(ANTIGENS,speeds)
plt.title("Antigen Speeds")
plt.show()
